## Instalacion de dependencias:

In [1]:
!pip install chromadb langchain-chroma langchain-huggingface sentence-transformers -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

## Import de librerias

In [3]:
import pandas as pd
import numpy as np
import time
import chromadb
from chromadb.config import Settings
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

## Carga de chunks y vectores

Se carga el CSV generado en el notebook 01 con todos los chunks y sus metadatos.
Los metadatos (chunk_id, fuente, tipo) permiten identificar la fuente de cada
fragmento al momento de responder preguntas.

In [5]:
df_chunks = pd.read_csv('chunks_reglamento.csv')
textos    = df_chunks['texto'].tolist()

metadatos = df_chunks[['chunk_id','fuente','tipo']].to_dict(orient='records')


In [9]:
print(f'chunks cargados: {len(textos)}')
print(f"Ejemplo metadata: {metadatos[0]}")


chunks cargados: 695
Ejemplo metadata: {'chunk_id': 0, 'fuente': 'RGUNALM_2023 (1).pdf', 'tipo': 'reglamento_universitario'}


## Carga de embeddings

Se usa el mismo modelo `nomic-embed-text-v1` del notebook 02.
Es fundamental usar el **mismo modelo** en toda la pipeline para que
los vectores de los chunks y los de las consultas sean comparables.
Si se usa un modelo diferente, las búsquedas darán resultados incorrectos.

In [10]:
embedder = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1",
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True}
)
print('Modelo listo---')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.3k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Modelo listo---


## Creacion de coleccion en ChromaDB

Se crea una colección con métrica **coseno** porque los embeddings están
normalizados (L2 norm = 1), lo que hace que coseno sea más estable que L2.


In [12]:
cliente = chromadb.PersistentClient(path="./chroma_reglamento")

nombre_coleccion = "reglamento_universitario"


try:
    cliente.delete_collection(nombre_coleccion)
    print("Colección anterior eliminada.")
except:
    pass

coleccion = cliente.create_collection(
    name=nombre_coleccion,
    metadata={"hnsw:space": "cosine"}
)

print(f"Colección '{nombre_coleccion}' creada con métrica coseno.")

Colección 'reglamento_universitario' creada con métrica coseno.


## Insercion de documentos en lotes

Los chunks se insertan en lotes de 100 para evitar errores de memoria en Colab.
Todos los valores de metadatos se convierten a string porque ChromaDB
lo requiere para almacenarlos correctamente.

In [13]:
BATCH_SIZE = 100
total      = len(textos)
ids        = [str(i) for i in range(total)]

print(f"Insertando {total} chunks en lotes de {BATCH_SIZE}...\n")
t0 = time.time()

for inicio in range(0, total, BATCH_SIZE):
    fin          = min(inicio + BATCH_SIZE, total)
    lote_textos  = textos[inicio:fin]
    lote_ids     = ids[inicio:fin]
    lote_meta    = metadatos[inicio:fin]

    # Convertir metadatos: todos los valores deben ser str o int
    lote_meta_limpio = [
        {k: str(v) for k, v in m.items()}
        for m in lote_meta
    ]

    coleccion.add(
        documents=lote_textos,
        ids=lote_ids,
        metadatas=lote_meta_limpio
    )

    print(f"  Insertados chunks {inicio} → {fin-1}")

tiempo_insert = time.time() - t0
print(f"\nInserción completa en {tiempo_insert:.1f}s")
print(f"Total en colección: {coleccion.count()}")

Insertando 695 chunks en lotes de 100...



/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 34.8MiB/s]


  Insertados chunks 0 → 99
  Insertados chunks 100 → 199
  Insertados chunks 200 → 299
  Insertados chunks 300 → 399
  Insertados chunks 400 → 499
  Insertados chunks 500 → 599
  Insertados chunks 600 → 694

Inserción completa en 111.7s
Total en colección: 695


## Verificacion de coleccion

In [21]:
print(f"Nombre:          {nombre_coleccion}")
print(f"Total documentos: {coleccion.count()}")

# Recuperar un documento de muestra
muestra = coleccion.get(ids=["53"], include=["documents","metadatas"])
print(f"  Texto:    {muestra['documents'][0][:200]}...")
print(f"  Metadata: {muestra['metadatas'][0]}")

Nombre:          reglamento_universitario
Total documentos: 695
  Texto:    REGLAMENTO GENERAL – Res. No. 0001-2017-AU-UNALM 
2017 
 
 
 
13 
ARTÍCULO 24°.- La unidad de extensión universitaria y proyección social de la facultad es 
la encargada de integrar las actividades de...
  Metadata: {'fuente': 'RGUNALM_2023 (1).pdf', 'tipo': 'reglamento_universitario', 'chunk_id': '53'}
